Elvato Development Documentation
===============================
Headless Ecommerce built-on Typescript/ React

Medusa development Admin credentials:
user: admin@elvato.com
pass: supersecret

CJ-Dropship User Credentials:
user: elvaolighting@gmail.com
pass: Msi58900!



# Development Structure Overview

<u>Structure</u>:
Storefront |
Database |
Admin |
Logic 

**Development**
1. Storefront <br> 
localhost:8889 <br>
1.1. Procurement Portal included as storefront page <br> 
localhost:8889/shipping <br>
User interaction interface <br>
Host: Vercel

2. Catelogue Manager <br>
localhost:8008 <br>
Wholesale API and UI for managing products, categories, collections, orders, customers, and inventory <br>
Host: Vercel

3. Admin <br>
localhost:9000/admin <br>
React/ Vite5 UI served BY backend which uses Admin API from backend <br>
Host: Vercel

4. Logic <br>
localhost:9000 <br>
Provides all Store API and Admin API endpoints <br>
Host: Render

5. Database Postgresql <br>
localhost:5432 <br>
Stores all ecommerce data <br>
Host: Neon

# Competitive Analysis & Research

-FinesseDecor: https://finessedecor.com/collections/chandeliers
traffic: 4,141
bounce-rate: 43.97%
sales: n/a
rating: sucks 
Modern Lighting focus, interesing curation of likeminded products. 

-LightsCanada: https://lightscanada.ca/
traffic: 87,706/ month
bounce-rate: 36.91%
sales: n/a 
rating: pretty good
Large selection of lighting with a variety of collections and style.

-Lumens: https://www.lumens.com/
traffic: 1.451M
bounce-rate: 52.14%
sales: $14.98MM/ month 
rating: excellent
A top online lighting and modern furniture retailer.

-2modern: https://www.2modern.com/
traffic: 459,150
bounce-rate: 48.87%
sales: $1.43MM/ month
rating: pretty good
An e‑commerce platform specializing in contemporary design furniture and lighting. 

-ShadesofLight: https://www.shadesoflight.com/
traffic: 516,465
bounce-rate: 42.75%
sales: $2.95MM/ month
rating: pretty good
A national catalog/online lighting store.

-YLighting: https://www.ylighting.com/ (dead/ Wayfair now)
traffic: 1,583
bounce-rate: 39.96%
sales: n/a
rating: good
A long‑time leader in contemporary lighting (now a Wayfair brand). 

-MontrealLighting: https://www.montreallighting.com/
traffic: 38,843
bounce-rate: 30.28%
sales: n/a
rating: fair
A Canadian online lighting retailer with a wide selection of products.



# CJ-API Docs

### General Overview

CJ Dropshipping API powers the **Inventory** tab in the Catalogue Manager, enabling:
- Product discovery and search across CJ's catalog
- Filtering by lighting-specific categories
- Importing products to local database (TODO)

### Integration Architecture

```
┌─────────────────┐      ┌──────────────────┐      ┌─────────────────┐
│  Catalogue UI   │ ──▶  │  Next.js API     │ ──▶  │  CJ API v2.0    │
│  (Inventory)    │      │  /api/cj/*       │      │  developers.cj  │
└─────────────────┘      └──────────────────┘      └─────────────────┘
```

### Our API Routes

| Route | Method | Purpose |
|-------|--------|---------|
| `/api/cj/auth` | GET | Check auth status |
| `/api/cj/auth` | POST | Force token refresh |
| `/api/cj/products` | GET | Search products with filters |
| `/api/cj/categories` | GET | Get category tree (cached 1hr) |

### Authentication Flow

| Step | Action | Token |
|------|--------|-------|
| 1 | Request with `CJ_API_KEY` | → Access Token (15 days) |
| 2 | Cache token server-side | — |
| 3 | Auto-refresh when expired | Refresh Token (180 days) |

### Search Parameters We Use

| Parameter | Type | Description |
|-----------|------|-------------|
| `keyWord` | string | Search term (e.g., "chandelier") |
| `categoryId` | string | 3rd-level category UUID |
| `page` / `size` | int | Pagination (max 100/page) |
| `orderBy` | int | 0=match, 1=popularity, 2=price, 3=date |
| `sort` | string | `asc` or `desc` |
| `startSellPrice` / `endSellPrice` | float | Price range (USD) |

### Lighting Presets

| Preset | Keywords |
|--------|----------|
| All Lighting | lamp, light, LED |
| Pendant & Chandeliers | pendant, chandelier, hanging |
| Table & Desk Lamps | table lamp, desk lamp |
| Wall Lights | wall light, sconce |
| Smart Lighting | smart, wifi, RGB |
| Decorative | fairy, string, neon |
| Outdoor | garden, solar, pathway |

### Response Structure

```
CJ API Response:
├── code: 200
├── data
│   ├── pageNumber, pageSize, totalRecords
│   └── content[]
│       └── [0]
│           ├── list[] ← Products array
│           ├── relatedCategoryList[]
│           └── keyWord
```

### Environment Variables

| Variable | Location | Required |
|----------|----------|----------|
| `CJ_API_KEY` | `/catalogue/.env.local` | ✓ |

## Rate Limits

### Authentication Limits
| Endpoint | Limit |
|----------|-------|
| `getAccessToken` | **Once per 5 minutes** |
| `refreshAccessToken` | 5 times per minute |

### Token Lifecycle
| Token Type | Validity |
|------------|----------|
| Access Token | 15 days |
| Refresh Token | 180 days |

### General Request Limits
| Limit Type | Rate |
|------------|------|
| Per IP address | 10 requests/second |
| Non-login interfaces | 30 requests/second |

### User Level Limits (Elvato = Level 3 Prime)
| User Level | Requests/Second |
|------------|-----------------|
| Free / Level 0-1 | 1 req/sec |
| Plus / Level 2 | 2 req/sec |
| **Prime / Level 3** | **4 req/sec** ✓ |
| Advanced / Level 4-5 | 6 req/sec |

### Product List API Specific
- Free/V1 users: Limited to 1,000 requests/day
- 1 IP → max 3 user accounts

---

## Key Endpoints

### Authentication
- `POST /api2.0/v1/authentication/getAccessToken` - Get new token
- `POST /api2.0/v1/authentication/refreshAccessToken` - Refresh token

### Products
- `GET /api2.0/v1/product/listV2` - Search products (Elasticsearch)
- `GET /api2.0/v1/product/query` - Get product details
- `GET /api2.0/v1/product/getCategory` - Get category tree

### Required Headers
```
Content-Type: application/json
CJ-Access-Token: <your-access-token>
```

---

## Implementation Notes

**Token Caching Issue (Development)**
- In dev mode, hot-reloads clear the in-memory token cache
- Each restart requires waiting 5 min for new token
- Solution: Don't edit files while testing CJ API

**API Key Location**
- Dashboard: https://www.cjdropshipping.com/myCJ.html#/apikey
- Store in: `/catalogue/.env.local` as `CJ_API_KEY`